# Sherwood Webb et al. 2020 model — UL and US baselines

Minimal driver for [`SherwoodWebb.stan`](SherwoodWebb.stan): runs both the UL (uniform-$\lambda$) and
US (uniform-$S$) baselines and compares the marginal posterior of $S$ to Sherwood Table 10.

*Notes*: 
- More information can be found in the Readme.md file in the repo
- The model has one minor difference from Sherwood Webb et al, in that it does not account for correlations between historical forcing $F_{hist}$ and $F_{2\times\text{CO}_2}$. Yet.  
- To do: 
  - add in diagnostics plot for the HMC sampler (prior/posterior predictive checks, divergence plots, etc)
  - reproduce the rest of Sherwood Webb et al Table 10. 

In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import cmdstanpy

ROOT = Path(os.getcwd())
assert (ROOT / 'SherwoodWebb.stan').exists(), f'stan fie not found in {ROOT}'

## Sherwood baseline values

Process: aggregate of 11 Gaussian feedback components (Table 1).
Historical: Table 5 BASELINE row.  Paleo: Tables 7 (LGM) and 8 (mPWP).

The only difference between the UL and US data files is the integer flag
`use_uniform_lambda_prior`.

In [ ]:
# INPUT PARAMETER BLOCK

# -------------Posterior sample size-------------
N_CHAINS=4
N_WARMUP=5000
N_SAMPLES=125000

#Timing: ~ 1 sec / 10,000 samples with 4 chains
# e.g. 520,000 samples (N=4 chains)x(5000 warmups + 125,000 samples) takes ~50 seconds

#scaling between 5-95 percentiles and sigma for a gaussian:
PCT = 1.645   # 5/95 -> 1.645 sigma

#-------------Baseline values from Sherwood (pyCmndStan takes in data as a dictionary)-------------
BASE = {
    # -------------Shared-parameters -------------
    # Forcing
    'mu_F2xCO2':       4.0,
    'sig_F2xCO2':      0.3,

    # State Dependence
    'mu_zeta':         0.06,
    'sig_zeta':        0.20,


    # -------------Process-------------
    'mu_lambda':      -1.30, 
    'sig_lambda':      0.44,
    
    # -------------Historical-------------
    'mu_T_hist':       1.03, 
    'sig_T_hist':     (1.03 - 0.89) / PCT,
    
    'mu_N_hist':       0.6,
    'sig_N_hist':     (0.6 - 0.3)   / PCT,
    
    'mu_F_hist':       1.83,
    'sig_F_hist_low': (1.83 - (-0.03)) / PCT,
    'sig_F_hist_high':(2.71 - 1.83)    / PCT,
    
    'mu_dlambda':      0.5,
    'sig_dlambda':    (0.5 - 0.0)   / PCT,
    
    # -------------LGM-------------
    'mu_T_LGM':       -5.0,  
    'sig_T_LGM':       1.0,
    
    'mu_F_LGM':       -6.15,
    'sig_F_LGM':       2.0,

    'mu_alpha':        0.1,
    'sig_alpha':       0.1,
    
    # -------------mid-Pliocene-------------
    'mu_T_plio':       3.0,  
    'sig_T_plio':      1.0,

    'mu_CO2_plio':     375.0,
    'sig_CO2_plio':    25.0,
    
    'mu_fCH4':         0.4,
    'sig_fCH4':        0.1,
    
    'mu_fESS':         0.5,  
    'sig_fESS':        0.25,
}

# flag for choosing uniform lambda or uniform S
data_UL = dict(BASE, use_uniform_lambda_prior=1)
data_US = dict(BASE, use_uniform_lambda_prior=0)

#initial guesses for the HMC samper
init = dict(S=3.0, F_2xCO2=4.0, zeta=0.06,
            F_z=0.0, N_hist=0.6, dlambda=0.5,
            T_LGM=-5.0, alpha=0.1,
            CO2_plio=375.0, fCH4=0.4, fESS=0.5)

# Sherwood Table 10 reference values: (P5, P17, P50, P83, P95, mean)
SHERWOOD_UL = (2.3, 2.6, 3.1, 3.9, 4.7, 3.2)
SHERWOOD_US = (2.4, 2.8, 3.5, 4.5, 5.7, 3.7)

## Compile the model

In [ ]:
model = cmdstanpy.CmdStanModel(stan_file=str(ROOT / 'SherwoodWebb.stan'))
print('binary:', model.exe_file)

## Sample both US and UL baselines

Note: adapt_delta=0.95 reduces the number of divergences. This is probably due to the shape of posterior when $\lambda-\Delta \lambda=0$, since it the number of divergences started increasing when the pattern effect term was added to the historical likelihood


In [ ]:
# WRite the data to json
# we'll write to versions, one with uniform lambda flag, one with uniform S flag/ 
cmdstanpy.write_stan_json('/tmp/baseline_UL.json', data_UL)
cmdstanpy.write_stan_json('/tmp/baseline_US.json', data_US)

common = dict(inits=init, chains=N_CHAINS,
              iter_warmup=N_WARMUP, iter_sampling=N_SAMPLES,
              adapt_delta=0.95, show_progress=False)

fit_UL = model.sample(data='/tmp/baseline_UL.json', **common)
fit_US = model.sample(data='/tmp/baseline_US.json', **common)

S_UL = fit_UL.draws_xr(vars=['S']).S.values.ravel()
S_US = fit_US.draws_xr(vars=['S']).S.values.ravel()

## Summary

In [ ]:
def line(label, S, fit, sherwood):
    p5, p17, p50, p83, p95 = np.percentile(S, [5, 17, 50, 83, 95])
    div = int(np.sum(fit.method_variables()['divergent__']))
    print(f'=== {label} ===')
    print(f'  n_draws       : {len(S)}')
    print(f'  mean +/- std  : {S.mean():.2f} +/- {S.std():.2f}')
    print(f'  divergences   : {div}')
    print()
    print(f'  First row: Sherwood Table 10 reference ({label}):')
    print(f'  Second row: Stan estimate ({label}):')
    print(f'    P5  P17  P50  P83  P95   mean')
    print(f'    {sherwood[0]:.2f} {sherwood[1]:.2f}  {sherwood[2]:.2f} {sherwood[3]:.2f}  {sherwood[4]:.2f}   {sherwood[5]:.2f}')
    print(f'    {p5:.2f} {p17:.2f}  {p50:.2f} {p83:.2f}  {p95:.2f}   {S.mean():.2f}')
    print()

line('UL baseline', S_UL, fit_UL, SHERWOOD_UL)
line('US baseline', S_US, fit_US, SHERWOOD_US)

## Plot

In [ ]:
S_grid = np.linspace(0.5, 8, 500)

fig, ax = plt.subplots(figsize=(11, 5.5))

for S, lbl, color, sw in [(S_UL, 'UL baseline', 'tab:blue',  SHERWOOD_UL),
                            (S_US, 'US baseline', 'tab:green', SHERWOOD_US)]:
    pdf = gaussian_kde(S, bw_method=0.1)(S_grid); pdf /= pdf.max()
    ax.plot(S_grid, pdf, '-', lw=2.5, color=color, label=f'Stan: {lbl}')
    ax.fill_between(S_grid, pdf, alpha=0.15, color=color)
    
    # bold dashed line at SW2- median
    ax.axvline(sw[2], color=color, ls='-', lw=1.4, alpha=0.85,
               label=f'SW20 {lbl} median ({sw[2]} K)')

    # bold dashed line at SW2- median
    ax.axvline(np.percentile(S,50), color=color, ls='--', lw=1.4, alpha=0.85,
               label=f'Stan {lbl} median ({sw[2]} K)')

ax.set_xlim(0, 8); ax.set_ylim(0, 1.15)
ax.set_xlabel('S (K)'); ax.set_ylabel('p(S | E) / max')
ax.set_title('Combined SW20 Bayesian posterior — UL and US baselines')
ax.grid(alpha=0.4); ax.legend(loc='upper right')
plt.tight_layout()